# Segmentación de municipios del Estado de México con KMeans

**Materia:** Extracción de Conocimiento en Bases de Datos  
**Fuente principal:** INEGI — Censo de Población y Vivienda 2020 (ITER)  
**Validación externa:** CONAPO — Índices de Marginación 2020

**Pregunta guía:** ¿Qué municipios del Estado de México presentan características similares en sus condiciones de vivienda, acceso a servicios y conectividad?

## Parte 1 — Imports y configuración

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.max_columns", 50)
SEED = 42

BASE = Path.cwd().parent
RAW = BASE / "data" / "raw" / "iter_00_cpv2020" / "conjunto_de_datos" / "conjunto_de_datos_iter_00CSV20.csv"
OUT = BASE / "data" / "processed" / "dataset_limpio.csv"

## Parte 2 — Carga del dataset original

El archivo crudo nacional (`iter_00_cpv2020_csv.zip`, ~35 MB) no se incluye en el repositorio. Debe descargarse desde:
https://www.inegi.org.mx/contenidos/programas/ccpv/2020/datosabiertos/iter/iter_00_cpv2020_csv.zip

y extraerse en `data/raw/`. El CSV usa codificación UTF-8 con BOM (`utf-8-sig`).

In [2]:
df_raw = pd.read_csv(RAW, dtype=str, encoding="utf-8-sig")
print("Filas totales nacionales:", len(df_raw))
print("Columnas:", df_raw.shape[1])
df_raw.head(3)

Filas totales nacionales: 195662
Columnas: 286


,ENTIDAD,NOM_ENT,MUN,NOM_MUN,LOC,NOM_LOC,LONGITUD,LATITUD,ALTITUD,POBTOT,POBFEM,POBMAS,P_0A2,P_0A2_F,P_0A2_M,P_3YMAS,P_3YMAS_F,P_3YMAS_M,P_5YMAS,P_5YMAS_F,P_5YMAS_M,P_12YMAS,P_12YMAS_F,P_12YMAS_M,P_15YMAS,...,VPH_C_SERV,VPH_NDEAED,VPH_DSADMA,VPH_NDACMM,VPH_SNBIEN,VPH_REFRI,VPH_LAVAD,VPH_HMICRO,VPH_AUTOM,VPH_MOTO,VPH_BICI,VPH_RADIO,VPH_TV,VPH_PC,VPH_TELEF,VPH_CEL,VPH_INTER,VPH_STVP,VPH_SPMVPI,VPH_CVJ,VPH_SINRTV,VPH_SINLTC,VPH_SINCINT,VPH_SINTIC,TAMLOC
0,00,Total nacional,000,Total nacional,0000,Total nacional,NaN,NaN,NaN,126014024,64540634,61473390,5764054,2848875,2915179,119976584,61554567,58422017,115693273,59433559,56259714,100528155,51962264,48565891,93985354,...,32671764,79584,32979844,16874580,581095,30811260,25610544,16651199,16340788,4227460,7469168,23772973,32031555,13204680,13184550,30775898,18307193,15211306,6616141,4047100,1788552,3170894,15108204,852871,*
1,00,Total nacional,000,Total nacional,9998,Localidades de una vivienda,NaN,NaN,NaN,250354,96869,153485,10493,5193,5300,239441,91463,147978,232086,87931,144155,207748,76111,131637,197411,...,35091,4842,43668,24373,5136,38199,26412,13608,30940,15001,13600,36738,40001,5797,3523,47005,8385,18981,1732,1113,12775,14143,51293,7154,*
2,00,Total nacional,000,Total nacional,9999,Localidades de dos viviendas,NaN,NaN,NaN,147125,61324,85801,6798,3407,3391,139757,57628,82129,135028,55256,79772,119223,47543,71680,111530,...,19807,2935,23841,16122,4115,21775,15880,7902,16699,8076,6330,20009,23198,3588,2177,25581,5027,11306,971,708,8247,10065,29741,5283,*


## Parte 3 — Revisión inicial de la estructura

Cada fila es una localidad. Los totales municipales son los registros con `LOC == "0000"`.

In [3]:
df_raw["NOM_LOC"].value_counts().head(8)

NOM_LOC
Total del Municipio             2469
Localidades de una vivienda     1990
Localidades de dos viviendas    1738
San Antonio                      647
Ninguno                          586
San José                         546
La Esperanza                     541
San Isidro                       536
Name: count, dtype: int64

## Parte 4 — Filtro: municipios del Estado de México

- `LOC == "0000"`: registro municipal.
- `ENTIDAD == "15"`: Estado de México.
- Se excluye la fila `MUN == "000"` (total de la entidad).

Se esperan exactamente **125 municipios**.

In [4]:
mun = df_raw[
    (df_raw["LOC"] == "0000")
    & (df_raw["ENTIDAD"] == "15")
    & (df_raw["MUN"] != "000")
].copy()
assert len(mun) == 125, f"Se esperaban 125 municipios, hay {len(mun)}"
mun[["MUN", "NOM_MUN", "POBTOT"]].head()

,MUN,NOM_MUN,POBTOT
90524,001,Acambay de Ruíz Castañeda,67872
90628,002,Acolman,171507
90661,003,Aculco,49266
90732,004,Almoloya de Alquisiras,15333
90771,005,Almoloya de Juárez,174587


## Parte 5 — Limpieza de valores especiales y tipos

El ITER usa `*` para valores no disponibles por redondeo o por ley de confidencialidad. Se convierten las columnas seleccionadas a numérico (`errors="coerce"`) y se verifica que no queden valores nulos.

In [5]:
id_cols = ["MUN", "NOM_MUN"]
desc_cols = ["POBTOT", "TVIVHAB", "VIVPAR_HAB", "VIVPARH_CV"]
vph_cols = ["VPH_AGUADV", "VPH_DRENAJ", "VPH_C_ELEC", "VPH_INTER", "VPH_PC", "VPH_CEL", "VPH_AUTOM"]
direct_cols = ["GRAPROES", "PROM_OCUP"]

df = mun[id_cols + desc_cols + vph_cols + direct_cols].copy()

for c in desc_cols + vph_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in direct_cols:
    df[c] = df[c].str.replace(",", ".", regex=False)
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Valores '*' en las columnas seleccionadas:", int((mun[desc_cols + vph_cols + direct_cols] == "*").sum().sum()))
print("Nulos tras conversión:", int(df.isna().sum().sum()))
df.describe().loc[["min", "max"]].round(2)

Valores '*' en las columnas seleccionadas: 0
Nulos tras conversión: 0


,POBTOT,TVIVHAB,VIVPAR_HAB,VIVPARH_CV,VPH_AGUADV,VPH_DRENAJ,VPH_C_ELEC,VPH_INTER,VPH_PC,VPH_CEL,VPH_AUTOM,GRAPROES,PROM_OCUP
min,4862.0,1234.0,1223.0,1216.0,1216.0,1192.0,1215.0,63.0,86.0,897.0,458.0,6.48,3.27
max,1645352.0,448688.0,432424.0,448068.0,443357.0,445737.0,447378.0,290588.0,202160.0,401562.0,186075.0,12.27,4.59


## Parte 6 — Selección de variables

Selección basada en el diccionario oficial (`referencias/fd_iter_cpv2020.pdf`):

| Variable | Significado | Rol |
|---|---|---|
| `VIVPARH_CV` | Viviendas particulares habitadas con características | Denominador |
| `VPH_AGUADV` | Agua entubada en la vivienda | Servicios |
| `VPH_DRENAJ` | Drenaje | Servicios |
| `VPH_C_ELEC` | Energía eléctrica | Servicios |
| `VPH_INTER` | Internet | Conectividad |
| `VPH_PC` | Computadora, laptop o tablet | Conectividad/equipamiento |
| `VPH_CEL` | Teléfono celular | Conectividad |
| `VPH_AUTOM` | Automóvil o camioneta | Equipamiento |
| `GRAPROES` | Grado promedio de escolaridad | Educación (directa) |
| `PROM_OCUP` | Promedio de ocupantes por vivienda | Densidad (directa) |

**Nota metodológica importante:** los conteos `VPH_*` se expresan respecto al universo de viviendas particulares habitadas **con características** (`VIVPARH_CV`), no respecto a `VIVPAR_HAB`. Al validar los datos se comprobó que dividir entre `VIVPAR_HAB` produce proporciones mayores a 1 en varios municipios, porque `VIVPARH_CV` es el universo real de esos indicadores según INEGI. `POBTOT` se conserva solo como variable descriptiva para evitar que el tamaño del municipio domine el clustering.

In [6]:
for c in vph_cols:
    df["p_" + c.replace("VPH_", "").lower()] = df[c] / df["VIVPARH_CV"]

prop_cols = [c for c in df.columns if c.startswith("p_")]
df[prop_cols].describe().loc[["min", "max", "mean"]].round(4)

,p_aguadv,p_drenaj,p_c_elec,p_inter,p_pc,p_cel,p_autom
min,0.7006,0.5777,0.9654,0.0375,0.0381,0.5414,0.2222
max,1.0000,0.9979,0.9994,0.7571,0.6131,0.9444,0.6390
mean,0.9583,0.9410,0.9921,0.3930,0.2820,0.8475,0.4074


## Parte 7 — Proporciones y verificación

Las proporciones deben quedar acotadas en [0, 1].

In [7]:
fuera_de_rango = int(((df[prop_cols] < 0) | (df[prop_cols] > 1)).sum().sum())
assert fuera_de_rango == 0, f"Proporciones fuera de rango: {fuera_de_rango}"
print("Todas las proporciones están en [0, 1]:", fuera_de_rango == 0)
df[["NOM_MUN"] + prop_cols].head()

Todas las proporciones están en [0, 1]: True


,NOM_MUN,p_aguadv,p_drenaj,p_c_elec,p_inter,p_pc,p_cel,p_autom
90524,Acambay de Ruíz Castañeda,0.896036,0.789828,0.973995,0.171049,0.142282,0.770266,0.362350
90628,Acolman,0.903105,0.988473,0.997770,0.499514,0.346840,0.909376,0.404363
90661,Aculco,0.928566,0.802830,0.975143,0.164054,0.119847,0.821721,0.454149
90732,Almoloya de Alquisiras,0.958423,0.896774,0.986858,0.224134,0.167742,0.845639,0.475747
90771,Almoloya de Juárez,0.925519,0.873486,0.987270,0.263293,0.204433,0.801019,0.363779


## Parte 8 — Guardado del dataset procesado

Salida: `data/processed/dataset_limpio.csv` (125 filas, una por municipio). Este archivo sí se sube al repositorio.

In [8]:
df.to_csv(OUT, index=False, encoding="utf-8-sig")
verif = pd.read_csv(OUT)
print("Guardado:", OUT)
print("Filas:", len(verif), "| Columnas:", len(verif.columns))
verif.head()

Guardado: C:\Users\corey\OneDrive\Desktop\Proyecto-Articulo\data\processed\dataset_limpio.csv
Filas: 125 | Columnas: 22


,MUN,NOM_MUN,POBTOT,TVIVHAB,VIVPAR_HAB,VIVPARH_CV,VPH_AGUADV,VPH_DRENAJ,VPH_C_ELEC,VPH_INTER,VPH_PC,VPH_CEL,VPH_AUTOM,GRAPROES,PROM_OCUP,p_aguadv,p_drenaj,p_c_elec,p_inter,p_pc,p_cel,p_autom
0,1,Acambay de Ruíz Castañeda,67872,17393,16912,17381,15574,13728,16929,2973,2473,13388,6298,8.30,3.90,0.896036,0.789828,0.973995,0.171049,0.142282,0.770266,0.362350
1,2,Acolman,171507,45406,41825,45286,40898,44764,45185,22621,15707,41182,18312,9.82,3.65,0.903105,0.988473,0.997770,0.499514,0.346840,0.909376,0.404363
2,3,Aculco,49266,13111,12731,13075,12141,10497,12750,2145,1567,10744,5938,8.02,3.76,0.928566,0.802830,0.975143,0.164054,0.119847,0.821721,0.454149
3,4,Almoloya de Alquisiras,15333,4191,4146,4185,4011,3753,4130,938,702,3539,1991,8.14,3.66,0.958423,0.896774,0.986858,0.224134,0.167742,0.845639,0.475747
4,5,Almoloya de Juárez,174587,42214,40020,42185,39043,36848,41648,11107,8624,33791,15346,8.68,4.03,0.925519,0.873486,0.987270,0.263293,0.204433,0.801019,0.363779
